# Sprint E9 walkthrough: transaction costs and capacity

In [1]:
# the repository root is importable so the package and the dashboard
# module can be imported without installing the wheel
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
COSTS = DATA / "costs"

In [2]:
# the data hash in the results file must be the hash of the artifacts
# the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads(
    (ROOT / "sprints" / "E9" / "RESULTS.json").read_text()
)
assert evaluate.e9_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])
print("verdicts:", stored["reference_values"]["verdicts"])

data_hash 9840953f472713c67d4e747928c6e0b5ef97b8cb1ebd7da433d2f4f4deb3ca98
verdicts: {'F9.1': 'fail', 'F9.2': 'fail', 'F9.3': 'not evaluable', 'F9.4': 'pass', 'F9.5': 'pass'}


## 1. Every criterion, its stored number and its verdict

In [3]:
for name, block in stored["criteria"].items():
    print(name, block["verdict"], json.dumps(block["stored_numbers"])[:160])

F9.1 fail {"n_monotonicity_violations_net_sharpe": 0, "n_monotonicity_violations_net_mean": 0, "n_curves": 9, "halving_aum_by_rho_k": {"0.02_0.25": NaN, "0.02_0.5": NaN, 
F9.2 fail {"0.02": {"turnover_cut": 0.37549626103844624, "ex_ante_ir_loss": 0.06485377957194484}, "0.05": {"turnover_cut": 0.3750665373197515, "ex_ante_ir_loss": 0.064753
F9.3 not evaluable {"spread_size_rank_correlation": NaN, "raw_cs_size_rank_correlation": -0.47129396780020943, "abdi_ranaldo_size_rank_correlation": 0.0031859126253798906, "cost_i
F9.4 pass {"0.02": {"k_0.25": NaN, "k_0.5": NaN, "k_1.0": NaN}, "0.05": {"k_0.25": 464158883.3612773, "k_0.5": 100000000.0, "k_1.0": 31622776.60168379}, "0.1": {"k_0.25":
F9.5 pass {"chosen_estimator": "size-decile schedule, 10 bp (smallest) to 1 bp (largest) half-spread, stated as an assumption", "median_half_spread_by_decile": {"0": 0.00


## 2. The Corwin-Schultz half-spread, by hand

beta and gamma from the high-low ranges; alpha = (sqrt(2 beta) - sqrt(beta)) / (3 - 2 sqrt(2)); the spread is 2 (exp(alpha) - 1) / (1 + exp(alpha)).

In [4]:
from efb import costs

prices = pd.read_parquet(DATA / "raw" / "prices.parquet")
spread = costs.corwin_schultz(prices)
print("names with a spread estimate", spread.notna().sum())
print("median half-spread", round(float(spread.median()), 6))
assert spread.notna().sum() > 100

names with a spread estimate 656
median half-spread 0.0


## 3. The capacity curve, live

Net Sharpe against AUM, per rho and impact coefficient; the halving AUM is stored per rho.

In [5]:
capacity = pd.read_parquet(COSTS / "capacity.parquet")
halving = pd.read_parquet(COSTS / "capacity_halving.parquet")
print(halving.to_string())
assert not capacity.empty

    rho     k  gross_sharpe   halving_aum
0  0.02  0.25      1.217569           NaN
1  0.02  0.50      1.217569           NaN
2  0.02  1.00      1.217569           NaN
3  0.05  0.25      2.830875  4.641589e+08
4  0.05  0.50      2.830875  1.000000e+08
5  0.05  1.00      2.830875  3.162278e+07
6  0.10  0.25      5.087021  2.154435e+09
7  0.10  0.50      5.087021  4.641589e+08
8  0.10  1.00      5.087021  1.000000e+08


## 4. The turnover versus IR trade-off (F9.2)

In [6]:
tradeoff = pd.read_parquet(COSTS / "turnover_tradeoff.parquet")
print(tradeoff.to_string())

    rho  turnover_cut  ex_ante_ir_loss
0  0.02      0.375496         0.064854
1  0.05      0.375067         0.064753
2  0.10      0.375387         0.064744


## 5. The D8 panel map

In [7]:
from dashboard.tabs import d08_costs as d8

curves = d8.load_cost_curves()
assert not curves.empty
print("D8 reads the cost curves, the capacity curve and the trade-off")

D8 reads the cost curves, the capacity curve and the trade-off


## 6. The memo's evidence, in citation order

In [8]:
memo = (ROOT / "docs" / "research" / "E9_tcost_capacity.md").read_text()
joined = " ".join(memo.split())
for name in ("F9.1", "F9.2", "F9.3", "F9.4"):
    assert name in joined, name
assert "What would falsify this?" in joined
assert "synthetic" in joined
print("memo cites every criterion and the falsification section")

memo cites every criterion and the falsification section


In [9]:
# closing checklist: every criterion name is covered by the code
import json as _json

source = "\n".join(
    "".join(cell["source"])
    for cell in _json.loads(
        (ROOT / "notebooks" / "E9_walkthrough.ipynb").read_text()
    )["cells"]
    if cell["cell_type"] == "code"
)
assert all(
    name in source for name in ("F9.1", "F9.2", "F9.3", "F9.4")
)
print("closing checklist: clean")

closing checklist: clean
